# Lesson 4 — Training a tiny BERT, Karpathy-style

Companion to [`04_transformer_and_mlm.py`](../04_transformer_and_mlm.py) and [`04_walkthrough.md`](../04_walkthrough.md).

This is the first lesson where we put **everything together**:

- **Embeddings** (L2) — words become vectors
- **Attention** (L3) — vectors look at each other
- **The 5-line training loop** (L1) — nudge the knobs until the loss drops

Today's task is **fill-in-the-blank** (a.k.a. **masked language modelling, MLM**). It's how BERT, PRAGMA, and every other "understand text" model is pre-trained.

> 🔑 The training loop is **the same** 5 lines from Lesson 1. The model has 1000× more parameters than the linear regression, but the loop doesn't change. That's the whole point.

---

## 🧰 Prerequisites

- [Lesson 1](lesson_01_linear_regression.ipynb) — the 5-line training loop, gradients, the basic recipe.
- [Lesson 2](../02_tokens_and_embeddings.py) — what an embedding is.
- [Lesson 3](../03_attention.py) — what attention does.


## Step 0 — Imports and a fixed seed

In [ ]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
random.seed(0)
torch.set_printoptions(precision=3, sci_mode=False)

print("torch version:", torch.__version__)

## Step 1 — The vocabulary and the secret rules

Three pets, three sounds. The vocabulary has just 8 tokens.

The **secret rules** are: `dog → bark`, `cat → meow`, `fish → swim`. The model does NOT see these rules. It only sees masked examples and has to figure them out from the data.

In [ ]:
vocab = ["<pad>", "<mask>", "dog", "cat", "fish", "bark", "meow", "swim"]
tok2id = {w: i for i, w in enumerate(vocab)}
V = len(vocab)

PAIRS = [("dog", "bark"), ("cat", "meow"), ("fish", "swim")]

print(f"Vocabulary size: V = {V}")
print()
print("Token table:")
print(f"  {'id':>3}  {'token':<8}")
for w, i in tok2id.items():
    print(f"  {i:>3}  {w}")
print()
print(f"Secret rules (model never sees these):")
for animal, sound in PAIRS:
    print(f"  {animal} → {sound}")

**Note on the two special tokens:**

- `<pad>` (id=0) — used to fill empty positions when sequences are different lengths. Not used in this lesson because all our examples are exactly 2 tokens.
- `<mask>` (id=1) — the "hidden" token. We replace one position in each training example with this, and ask the model to guess what was originally there.

`<mask>` is the heart of MLM training. Without it, there's nothing for the model to predict.

## Step 2 — Build one training example by hand

Before we automate, let's make ONE example to understand the shape.

Pick a pair: `("dog", "bark")`. Encode as ids: `[2, 5]`. Now hide ONE position (let's hide position 1, the sound). The masked input becomes `[2, 1]` (where `1` is `<mask>`). The training label is the original token at that position — `5` (which is `bark`).

In [ ]:
animal, sound = "dog", "bark"
ids = [tok2id[animal], tok2id[sound]]
print(f"Original ids:    {ids}    ← {animal} {sound}")

# We'll mask position 1 (the sound)
mask_position = 1
true_value = ids[mask_position]   # remember the answer
ids[mask_position] = tok2id["<mask>"]
print(f"Masked input:    {ids}    ← {vocab[ids[0]]} {vocab[ids[1]]}")
print(f"True answer at position {mask_position}: token id {true_value} ({vocab[true_value]})")
print()
print("The model's job: see 'dog <mask>', predict 'bark' at position 1.")

## Step 3 — A function that makes random examples

For training we need many examples. Each one:
1. Picks a random `(animal, sound)` pair.
2. Picks a random position (0 or 1) to mask.
3. Returns: `(input_ids, labels)` where labels has the truth at the masked position and `-100` everywhere else.

`-100` is a special "ignore me" value. PyTorch's `CrossEntropyLoss` skips it. This lets the loss focus only on the position we want the model to predict.

In [ ]:
def make_example():
    animal, sound = random.choice(PAIRS)
    ids = [tok2id[animal], tok2id[sound]]
    labels = [-100, -100]                       # placeholder
    hide_position = random.choice([0, 1])
    labels[hide_position] = ids[hide_position]   # the answer at the masked position
    ids[hide_position] = tok2id["<mask>"]        # actually mask it
    return ids, labels

# Generate 5 examples to see what they look like
print(f"{'input ids':<20s}  {'input tokens':<22s}  {'labels':<20s}  {'meaning':<30s}")
print("-" * 100)
for _ in range(5):
    ids, labels = make_example()
    tokens = [vocab[i] for i in ids]
    # Decode the label (the masked position has the real token id)
    target = next((vocab[l] for l in labels if l != -100), "?")
    visible = next(t for t in tokens if t != "<mask>")
    print(f"{str(ids):<20s}  {str(tokens):<22s}  {str(labels):<20s}  see {visible}, predict {target}")

**Key observation:** each example randomly masks position 0 or position 1. This is important — the model has to learn the relationship **in both directions**:
- Given `dog`, predict `bark`
- Given `bark`, predict `dog`

If we always masked position 1, the model would only learn the "forward" direction.

## Step 4 — Cross-entropy loss, by hand

For a single example, **cross-entropy loss** measures how surprised the model was by the correct answer. The formula:

$$\mathrm{loss} = -\log(P_{\text{model}}(\text{correct token}))$$

In English: "the negative log of the probability the model assigned to the right answer."

- If the model says "I'm 100% sure it's `bark`", and `bark` IS the answer: loss = `-log(1.0) = 0`. Perfect.
- If the model says "I'm 50/50 between `bark` and `meow`": loss = `-log(0.5) = 0.69`. Mediocre.
- If the model says "I'm 1% sure it's `bark`" and `bark` IS the answer: loss = `-log(0.01) = 4.6`. Terrible.

We'll compute this by hand on a fake "model output", verify against PyTorch, then move on.

In [ ]:
import math

# Fake model output: scores (logits) for each of the V vocab tokens.
# Suppose at position 1, our model outputs these scores:
logits_at_mask = torch.tensor([0.1, -0.5, 0.0, 0.2, -0.1, 2.5, 1.8, 0.4])
#                              <pad>, <mask>, dog, cat, fish, bark, meow, swim

# Convert scores to probabilities using softmax
probs = F.softmax(logits_at_mask, dim=-1)
print(f"{'token':<8s}  {'logit':>7s}  {'probability':>11s}")
print("-" * 30)
for i, tok in enumerate(vocab):
    bar = "█" * int(probs[i].item() * 30)
    print(f"{tok:<8s}  {logits_at_mask[i].item():>7.2f}  {probs[i].item():>11.4f}  {bar}")
print()

# The true answer is bark (id=5).
true_id = 5
p_true = probs[true_id].item()
loss_manual = -math.log(p_true)
print(f"P(model says 'bark') = {p_true:.4f}")
print(f"Loss = -log({p_true:.4f}) = {loss_manual:.4f}")
print()

# Verify with PyTorch
loss_torch = F.cross_entropy(logits_at_mask.unsqueeze(0), torch.tensor([true_id])).item()
print(f"PyTorch cross_entropy: {loss_torch:.4f}")
assert abs(loss_manual - loss_torch) < 1e-4, "manual != PyTorch"
print()
print("✓ Our hand-computed cross-entropy matches PyTorch's.")

**Interpretation:** the model put **66.3%** of its probability mass on `bark` (the right answer). It's mostly right, but not certain. Loss = 0.41.

After training, we want the model to be near 100% confident on the correct answer, so loss approaches 0.

## Step 5 — Build the tiny model

Three pieces. They should look familiar from Lessons 2-3.

```
ids ─► Embedding ─► TransformerEncoderLayer ─► Linear (MLM head) ─► scores
```

- **`Embedding(V=8, d=16)`** — turns each token id into a 16-dim vector. 128 trainable numbers.
- **`TransformerEncoderLayer(d=16, heads=2, ff=32)`** — attention + feed-forward + residual + LayerNorm. We saw exactly what's in this in [Lesson 4e](../04e_encoder_layer_from_scratch.py). ~3000 trainable numbers.
- **`Linear(d=16, V=8)`** — projects each contextualised vector back to scores over the 8 vocab tokens. 136 trainable numbers.

In [ ]:
class TinyMLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb  = nn.Embedding(V, 16)
        self.enc  = nn.TransformerEncoderLayer(
            d_model=16, nhead=2, dim_feedforward=32, batch_first=True,
        )
        self.head = nn.Linear(16, V)

    def forward(self, x):
        h = self.emb(x)            # ids -> vectors
        h = self.enc(h)            # vectors -> contextualised vectors
        return self.head(h)        # vectors -> vocab scores

model = TinyMLM()
total = sum(p.numel() for p in model.parameters())
print(f"Total trainable parameters: {total}")
print()

# Inspect the architecture
print("Layer breakdown:")
for name, p in model.named_parameters():
    shape_str = str(tuple(p.shape))
    print(f"  {name:<35s}  shape {shape_str:<15s}  {p.numel()} params")

**Compare to Lesson 1.** That model had **2 parameters**. This one has 3000+. The training loop is unchanged. That's the foundation-model bet — same loop scales to billions of parameters.

## Step 6 — What does the model say BEFORE training?

A randomly-initialised model has no idea about anything. Its outputs should be roughly uniform across the vocab. Let's verify.

In [ ]:
# Take a masked example
ids = torch.tensor([[tok2id["dog"], tok2id["<mask>"]]])  # batch of 1, length 2
print(f"Input (masked):    {ids.tolist()}    → ['dog', '<mask>']")
print()

with torch.no_grad():
    logits = model(ids)              # shape (1, 2, V)
    probs_mask = F.softmax(logits[0, 1], dim=-1)

print(f"Logits at the mask position, shape {str(tuple(logits[0, 1].shape))}:")
print()
print(f"{'token':<8s}  {'probability':>11s}  {'bar (untrained model — should be roughly uniform)':<60s}")
print("-" * 80)
for i, tok in enumerate(vocab):
    p = probs_mask[i].item()
    bar = "█" * int(p * 60)
    print(f"{tok:<8s}  {p:>11.3f}  {bar}")
print()
print(f"Probability spread: max - min = {(probs_mask.max() - probs_mask.min()).item():.3f}")
print("(Should be small — model is just guessing uniformly.)")

**Interpretation:** before training, the model's predictions are roughly uniform — each vocab token gets about 1/8 = 12.5% probability. There's no pattern yet.

Now we train.

## Step 7 — Train, with predictions inspected at checkpoints

Like in Lesson 1, we'll capture the model's behaviour at multiple checkpoints during training. This makes the learning **visible**.

Specifically, at steps 0, 50, 100, 200, 500, we'll fix a test input (`dog <mask>`) and record the model's probability distribution over the vocab. We want to watch `P(bark)` climb toward 1.0 and the others drop toward 0.

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

# Fixed test input: see 'dog', predict at mask position
test_ids = torch.tensor([[tok2id["dog"], tok2id["<mask>"]]])

# Track loss and the prob assigned to 'bark' over time
history = {"step": [], "loss": [], "p_bark": [], "p_meow": [], "p_swim": [], "snapshots": {}}

def snapshot(step):
    with torch.no_grad():
        logits = model(test_ids)
        probs  = F.softmax(logits[0, 1], dim=-1)
    history["snapshots"][step] = probs.clone()
    return probs

# Initial snapshot
snapshot(0)

# Train
N_STEPS = 500
BATCH = 32
for step in range(N_STEPS + 1):
    batch = [make_example() for _ in range(BATCH)]
    ids, labels = zip(*batch)
    x = torch.tensor(ids)
    y = torch.tensor(labels)

    logits = model(x)
    loss = loss_fn(logits.reshape(-1, V), y.reshape(-1))
    opt.zero_grad()
    loss.backward()
    opt.step()

    if step in [0, 10, 25, 50, 100, 200, 500]:
        probs = snapshot(step)
        history["step"].append(step)
        history["loss"].append(loss.item())
        history["p_bark"].append(probs[tok2id["bark"]].item())
        history["p_meow"].append(probs[tok2id["meow"]].item())
        history["p_swim"].append(probs[tok2id["swim"]].item())

# Print the evolution
print(f"Watching the model learn 'dog → bark':")
print()
print(f"{'step':>5}  {'loss':>7}  {'P(bark)':>9}  {'P(meow)':>9}  {'P(swim)':>9}  {'verdict':<40s}")
print("-" * 90)
for i, step in enumerate(history["step"]):
    verdict = ""
    if history["p_bark"][i] < 0.2:    verdict = "← still random"
    elif history["p_bark"][i] < 0.5:  verdict = "← starting to learn"
    elif history["p_bark"][i] < 0.9:  verdict = "← getting confident"
    else:                              verdict = "← nailed it!"
    print(f"{step:>5d}  {history['loss'][i]:>7.4f}  {history['p_bark'][i]:>9.3f}  {history['p_meow'][i]:>9.3f}  {history['p_swim'][i]:>9.3f}  {verdict}")

**Read the table top-to-bottom.** At step 0, `P(bark)` is around 12% (random). By step 100 it's climbing. By step 500 it's at 99%+ — the model is highly confident and correct.

Meanwhile `P(meow)` and `P(swim)` drop toward zero. The model has learned that **when the visible token is `dog`, the answer is `bark`** — not the other sounds.

## Step 8 — Visualise the probability evolution (ASCII)

In [ ]:
print("How the model's prediction for 'dog → ?' evolves during training:")
print()
print("           " + " ".join(f"{vocab[i]:<8s}" for i in range(V)))
for step, probs in history["snapshots"].items():
    bar_row = []
    for i in range(V):
        p = probs[i].item()
        # 5 chars to represent probability
        n = int(p * 5)
        bar = "█" * n + "·" * (5 - n)
        bar_row.append(bar)
    print(f"step {step:>4}  " + " ".join(f"{b:<8s}" for b in bar_row))
print()
print("Each row is the model's probability distribution at that training step.")
print("The 'bark' column starts uniform and grows to full bars by the end.")

## Step 9 — Test the trained model on each rule

We've seen `dog → bark` works. Let's verify all three rules and the reverse direction (`bark → dog`).

In [ ]:
def predict(visible_token, mask_position):
    """Given a visible token, predict the masked token at the given position."""
    ids = [tok2id["<mask>"], tok2id["<mask>"]]
    visible_position = 1 - mask_position
    ids[visible_position] = tok2id[visible_token]
    ids_t = torch.tensor([ids])
    with torch.no_grad():
        logits = model(ids_t)
        probs = F.softmax(logits[0, mask_position], dim=-1)
        top_id = logits[0, mask_position].argmax().item()
    return vocab[top_id], probs[top_id].item()

print("Forward direction (see animal, predict sound):")
for animal, sound in PAIRS:
    pred, conf = predict(animal, mask_position=1)
    status = "✓" if pred == sound else "✗"
    print(f"  {status} {animal:<5s} → predicted {pred:<5s} (confidence {conf*100:5.1f}%)   true: {sound}")
print()
print("Reverse direction (see sound, predict animal):")
for animal, sound in PAIRS:
    pred, conf = predict(sound, mask_position=0)
    status = "✓" if pred == animal else "✗"
    print(f"  {status} {sound:<5s} → predicted {pred:<5s} (confidence {conf*100:5.1f}%)   true: {animal}")

**The model has learned the bidirectional mapping** between animals and sounds with high confidence. This is the whole point of MLM training: by randomly masking either position, the model learns to predict in both directions, encoding the relationship in its embeddings.

## Step 10 — What did the embeddings learn?

This is one of the magical things about MLM training: the **embedding vectors** for related tokens end up close to each other in vector space, even though we never told the model they should.

Let's check: after training, are `dog`'s embedding and `bark`'s embedding similar?

In [ ]:
emb = model.emb.weight.detach()

def cosine_sim(a, b):
    return (a @ b / (a.norm() * b.norm())).item()

print("Cosine similarity between trained embeddings:")
print()
pairs_to_check = [
    ("dog",  "bark"),   # should be high (related!)
    ("cat",  "meow"),   # should be high
    ("fish", "swim"),   # should be high
    ("dog",  "meow"),   # should be lower (unrelated)
    ("cat",  "bark"),   # should be lower
    ("dog",  "cat"),    # how does the model see these? Maybe similar (both pets)
    ("bark", "meow"),   # also unclear — both sounds
]
for a, b in pairs_to_check:
    sim = cosine_sim(emb[tok2id[a]], emb[tok2id[b]])
    bar = "█" * int(abs(sim) * 30)
    print(f"  {a:<5s} · {b:<5s} = {sim:+.3f}    {bar}")

**Read the table:** the related pairs (animal + its sound) have high positive similarity. The unrelated pairs have lower (or negative) similarity. The model has spatially organized its embedding space to reflect which tokens go together.

**This emergent structure** is why pre-training works for downstream tasks. The embeddings encode useful relationships that any subsequent classifier can leverage.

## Step 11 — Cross-entropy on a final test, with shape inspection

Let's do one more full forward pass and inspect every tensor's shape. Useful as a sanity check pattern for any model.

In [ ]:
batch = [make_example() for _ in range(4)]
ids, labels = zip(*batch)
x = torch.tensor(ids)
y = torch.tensor(labels)

print(f"Batch of inputs:")
print(f"  x.shape: {tuple(x.shape)}   ← (batch=4, sequence_length=2)")
print(f"  x:")
for row in x.tolist():
    print(f"    {row}  → {[vocab[i] for i in row]}")
print()

# Forward pass with shape inspection at every step
h = model.emb(x)
print(f"After embedding:        h.shape = {str(tuple(h.shape)):<15s}    ← (B, L, d_model=16)")

h = model.enc(h)
print(f"After encoder:          h.shape = {str(tuple(h.shape)):<15s}    ← same shape, contextualised")

logits = model.head(h)
print(f"After head:             logits.shape = {str(tuple(logits.shape)):<15s}    ← (B, L, V=8)")
print()

# Compute the loss
loss = loss_fn(logits.reshape(-1, V), y.reshape(-1))
print(f"Loss on this batch: {loss.item():.4f}")
print(f"(Should be small — model is well-trained.)")

## Step 12 — Things to try

Progressive Karpathy-style exercises.

### 🟢 Easy

1. **Train less.** Change `N_STEPS` to 50. Re-run from Step 7. The model will be undertrained — look at the predictions table and see them improve more slowly.

2. **Inspect a different pair's evolution.** In Step 7, change `test_ids` to `[[tok2id["cat"], tok2id["<mask>"]]]`. Track `P(meow)`, `P(bark)`, `P(swim)` instead. Same story should emerge.

3. **Print the first 10 examples from `make_example()`.** Verify the distribution is balanced: roughly 1/6 of each (animal, sound, position) combination.

### 🟡 Medium

4. **Add a new pair.** Add `("bird", "tweet")` to `PAIRS` and update `vocab`. Re-run training. The model should learn the new pair without forgetting the old ones. (You may need more training steps.)

5. **Compute mean-attention.** In Step 7's loop, also save `model.enc.self_attn` output for the test input. After training, plot how the attention at the mask position evolves — does it learn to attend hard at the visible token?

6. **Single-token vocab attack.** Try `test_ids = [[tok2id["<mask>"], tok2id["<mask>"]]]` (both positions masked). What does the model predict? With no context, it should fall back to a uniform prior — or maybe the most frequent token. Inspect.

### 🔴 Hard (forward-pointers)

7. **Use your own MultiHeadAttention.** Replace `nn.TransformerEncoderLayer` with the hand-rolled version from [Lesson 4e](../04e_encoder_layer_from_scratch.py). Train both and verify the loss curve and final accuracy match.

8. **Scale up.** Increase the vocabulary to ~20 tokens with 5-7 secret pairs. Bump model size (d=32, more heads, more layers). Train longer. Watch performance scale.

9. **Move to a richer dataset.** Apply this exact training loop to `pragma_mini.py`'s data (3-key events: pet/action/place). The model and loss are identical — you've already built it. See [Lesson 5](../notebooks/lesson_05_pragma_mini.ipynb) for that.

## Summary

You just:

- ✅ Built a tiny BERT-style model (embedding + Transformer encoder layer + linear head, ~3000 parameters).
- ✅ Computed cross-entropy loss **by hand** and matched PyTorch.
- ✅ Generated masked language modelling examples.
- ✅ Trained the model with the **same 5-line loop** as Lesson 1's linear regression — just more parameters.
- ✅ Watched the model's probability for the correct answer climb from random (~12%) to confident (~99%) during training.
- ✅ Verified the trained embeddings encode the underlying relationships (related tokens have higher cosine similarity).
- ✅ Tested in both directions: `dog → bark` AND `bark → dog`.

**The recipe never changes.** This is the same loop, the same loss, the same optimiser as Lesson 1. The only thing different is the model in the middle. Lesson 5 will use the same recipe on a 14-token vocab; PRAGMA uses it on 28,000+ tokens with 1 billion parameters. **Same loop. Bigger model.**

Move on to [**Lesson 5**](lesson_05_pragma_mini.ipynb) — applying the recipe to `pragma_mini.py`'s 3-field events.